In [5]:
from __future__ import print_function
from __future__ import division

import platform
import numpy as np
import config2 as config

import socket
_sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
_gamma = np.load(config.GAMMA_TABLE_PATH)
"""Gamma lookup table used for nonlinear brightness correction"""

_prev_pixels = np.tile(253, (3, config.N_PIXELS))
"""Pixel values that were most recently displayed on the LED strip"""

pixels = np.tile(1, (3, config.N_PIXELS))
"""Pixel values for the LED strip"""

_is_python_2 = int(platform.python_version_tuple()[0]) == 2

def _update_esp8266():
    """Sends UDP packets to ESP8266 to update LED strip values

    The ESP8266 will receive and decode the packets to determine what values
    to display on the LED strip. The communication protocol supports LED strips
    with a maximum of 256 LEDs.

    The packet encoding scheme is:
        |i|r|g|b|
    where
        i (0 to 255): Index of LED to change (zero-based)
        r (0 to 255): Red value of LED
        g (0 to 255): Green value of LED
        b (0 to 255): Blue value of LED
    """
    global pixels, _prev_pixels
    # Truncate values and cast to integer
    pixels = np.clip(pixels, 0, 255).astype(int)
    # Optionally apply gamma correc tio
    p = _gamma[pixels] if config.SOFTWARE_GAMMA_CORRECTION else np.copy(pixels)
    MAX_PIXELS_PER_PACKET = 126
    # Pixel indices
    idx = range(pixels.shape[1])

    idx = [i for i in idx if not np.array_equal(p[:, i], _prev_pixels[:, i])]
    #print(idx)
    n_packets = len(idx) // MAX_PIXELS_PER_PACKET + 1
    idx = np.array_split(idx, n_packets)
    for packet_indices in idx:
        m = '' if _is_python_2 else []
        for i in packet_indices:
            if _is_python_2:
                m += chr(i) + chr(p[0][i]) + chr(p[1][i]) + chr(p[2][i])
            else:
                m.append(i)  # Index of pixel to change
                m.append(p[0][i])  # Pixel red value
                m.append(p[1][i])  # Pixel green value
                m.append(p[2][i])  # Pixel blue value
        m = m if _is_python_2 else bytes(m)
        _sock.sendto(m, (config.UDP_IP, config.UDP_PORT))
    _prev_pixels = np.copy(p)
    

In [21]:
import pyaudio
import time

import pyaudio
import numpy as np
import time

class NuPyaudio:

    def __init__(self, config):
        self.p = pyaudio.PyAudio()
        self.frames_per_buffer = int(config.MIC_RATE / config.FPS)
        
    def audio_stream(self):
        stream = self.p.open(
            format=pyaudio.paInt16,
            channels=1,
            rate=config.MIC_RATE,
            input=True,
            frames_per_buffer=self.frames_per_buffer)
        return stream
    
    def close_stream(self, stream):
        stream.stop_stream()
        stream.close()
        self.p.terminate()

    def run_stream(self):
        stream = self.audio_stream()
        overflows = 0
        prev_ovf_time = time.time()

        try:
            while True:
                try:
                    # Non-blocking read to avoid delay
                    y = np.frombuffer(stream.read(self.frames_per_buffer, exception_on_overflow=False), dtype=np.int16)
                    y = y.astype(np.float32)
                    # Avoid unnecessary reads (remove second stream.read)
                    available = stream.get_read_available()
                    if available > 0:
                        stream.read(available, exception_on_overflow=False)
                    # Yield the audio data
                    yield y

                except IOError:
                    overflows += 1
                    if time.time() > prev_ovf_time + 1:
                        prev_ovf_time = time.time()
                        print(f'Audio buffer has overflowed {overflows} times')
        finally:
            self.close_stream(stream)


def audio_visualizer():
    """Simple visualizer: Map audio volume to LED brightness."""
    print("Starting audio visualizer...")
    try:
        while True:
            # Get audio data from microphone
            audio = NuPyaudio(config)
            for data in audio.run_stream():
                # Convert raw audio bytes to NumPy array
            
                audio_data = np.frombuffer(data, dtype=np.int16)

          # Calculate volume (RMS) from audio data
            volume = np.sqrt(np.mean(np.square(audio_data)))

            # Scale volume to the LED range [0, 255]
            brightness = int(np.interp(volume, [0, 5000], [0, 255]))

            # Set pixel colors based on brightness
            pixels[0, :] = brightness  # Red channel
            pixels[1, :] = 255 - brightness  # Green channel (inverse)
            pixels[2, :] = (brightness // 2)  # Blue channel (half brightness)

            # Roll the pixels to create movement
            pixels = np.roll(pixels, 1, axis=1)

            # Update the LED strip
            update()

            # Control the update speed (adjust as needed)
            time.sleep(0.05)

    except KeyboardInterrupt:
        print("Stopping visualizer...")
    finally:
        # Turn off all LEDs before exiting
        pixels *= 0
        update()
        stream.stop_stream()
        stream.close()
        audio.terminate()
        _sock.close()

# Run the visualizer
if __name__ == '__main__':
    audio_visualizer()




Starting audio visualizer...
Stopping visualizer...


UnboundLocalError: cannot access local variable 'pixels' where it is not associated with a value

In [29]:
import numpy as np
import pyaudio
import time
import socket
import config  # Assume UDP_IP and UDP_PORT are defined here

# LED and audio setup
NUM_PIXELS = 50  # Adjust according to your LED strip size
pixels = np.zeros((3, NUM_PIXELS))  # Each pixel has RGB values
_prev_pixels = np.zeros_like(pixels)  # Store previous pixel states

# Initialize UDP socket for sending LED updates
_sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

# Audio configuration
CHUNK = 1024  # Number of audio samples per buffer
FORMAT = pyaudio.paInt16  # 16-bit audio
CHANNELS = 1  # Mono audio
RATE = 44100  # Sample rate in Hz

# Initialize PyAudio
audio = pyaudio.PyAudio()
stream = audio.open(format=FORMAT, channels=CHANNELS, rate=RATE, 
                    input=True, frames_per_buffer=CHUNK)
pixels = np.zeros((3, 3), dtype=int)

def update():
    """Send pixel data to ESP8266 over UDP."""
    global pixels, _prev_pixels
    pixels = np.clip(pixels, 0, 255).astype(int)
    changed_indices = np.where(~np.all(pixels == _prev_pixels, axis=0))[0]
    for i in changed_indices:
        message = bytearray([i, pixels[0, i], pixels[1, i], pixels[2, i]])
        _sock.sendto(message, (config.UDP_IP, config.UDP_PORT))
    _prev_pixels = np.copy(pixels)

def audio_visualizer():
    global pixels
    try:
        while True:
            # Simulate volume data (replace with actual audio data processing)
            volume = np.random.randint(0, 5000)

            # Scale volume to the LED range [0, 255]
            brightness = int(np.interp(volume, [0, 5000], [0, 255]))

            # Set pixel colors based on brightness
            pixels[0, :] = brightness  # Red channel
            pixels[1, :] = 255 - brightness  # Green channel (inverse)
            pixels[2, :] = (brightness // 2)  # Blue channel (half brightness)

            # Roll the pixels to create movement
            pixels = np.roll(pixels, 1, axis=1)

            # Update the LED strip
            update()

            # Control the update speed (adjust as needed)
            time.sleep(0.05)

    except KeyboardInterrupt:
        print("Stopping visualizer...")
    finally:
        # Turn off all LEDs before exiting
        pixels *= 0
        update()
        # Assuming stream and audio are defined elsewhere
        stream.stop_stream()
        stream.close()
        audio.terminate()
        _sock.close()

# Run the visualizer
if __name__ == '__main__':
    audio_visualizer()

ValueError: operands could not be broadcast together with shapes (3,3) (3,50) 

In [ ]:
pi